Install & Import Libraries

In [1]:
! pip install transformers datasets scikit-learn torch

In [2]:
import pandas as pd # For data manipulation
import numpy as np # For numerical operations
import torch # For deep learning    
from sklearn.model_selection import train_test_split # For splitting the dataset into training and validation sets
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix # For evaluating the model's performance
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments # For using pre-trained BERT and fine-tuning it on our dataset   

Load Dataset

In [3]:
# Download the dataset from the provided URL
!wget https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv

--2026-04-09 18:20:29--  https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66212309 (63M) [text/plain]
Saving to: ‘IMDB-Dataset.csv’

IMDB-Dataset.csv    100%[===================>]  63.14M   258MB/s    in 0.2s    

2026-04-09 18:20:31 (258 MB/s) - ‘IMDB-Dataset.csv’ saved [66212309/66212309]



In [4]:
df = pd.read_csv("IMDB-Dataset.csv")  # Load the dataset
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Preprocessing

In [5]:
# Convert labels
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0}) # Convert labels to binary

# Remove missing values
df.dropna(inplace=True) # Remove rows with missing values

# Basic cleaning
df['review'] = df['review'].str.lower() # Convert to lowercase

Train-Test Split

In [6]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.3, random_state=42) # Split into training and temp (validation + test)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42) # Split temp into validation and test

Tokenization (BERT)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # Tokenize the texts

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True) # Tokenize training texts
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True) # Tokenize validation texts
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True) # Tokenize test texts

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset Class (PyTorch)

In [8]:
class Dataset(torch.utils.data.Dataset): # Create a custom dataset class for PyTorch
    def __init__(self, encodings, labels): # Initialize the dataset with encodings and labels
        self.encodings = encodings # Store the encodings
        self.labels = list(labels) # Store the labels as a list

    def __getitem__(self, idx): # Get an item by index
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} # Convert encodings to tensors
        item['labels'] = torch.tensor(self.labels[idx]) # Add the label to the item
        return item # Return the item

    def __len__(self): # Get the length of the dataset
        return len(self.labels) # Return the number of samples in the dataset

train_dataset = Dataset(train_encodings, train_labels) # Create the training dataset
val_dataset = Dataset(val_encodings, val_labels) # Create the validation dataset
test_dataset = Dataset(test_encodings, test_labels) # Create the test dataset

Load BERT Model

In [9]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2) # Load the pre-trained BERT model for sequence classification with 2 labels

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Setup

In [16]:
training_args = TrainingArguments( # Set up training arguments for the Trainer
    output_dir='./results', # Directory to save the model and results
    learning_rate=2e-5, # Learning rate for training
    per_device_train_batch_size=8, # Batch size for training
    per_device_eval_batch_size=8, # Batch size for evaluation
    num_train_epochs=1, # Number of epochs to train
    do_eval=True, # Whether to evaluate the model during training
    optim="adamw_torch", # Optimizer to use
    logging_dir='./logs', # Directory to save logs
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Metrics Function

In [17]:
def compute_metrics(p): # Function to compute evaluation metrics
    preds = np.argmax(p.predictions, axis=1) # Get the predicted labels by taking the argmax of the predictions
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='binary') # Compute precision, recall, and F1 score
    acc = accuracy_score(p.label_ids, preds) # Compute accuracy
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

Trainer

In [18]:
trainer = Trainer( # Initialize the Trainer with the model, training arguments, datasets, and metrics
    model=model, # The model to be trained
    args=training_args, # The training arguments
    train_dataset=train_dataset, # The training dataset
    eval_dataset=val_dataset, # The validation dataset
    compute_metrics=compute_metrics, # The function to compute evaluation metrics
)

Train Model

In [19]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [20]:
trainer.train() # Train the model

Step,Training Loss
500,0.344849
1000,0.293650
1500,0.297863
2000,0.266088
2500,0.247276
3000,0.249928
3500,0.239391
4000,0.217614


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4375, training_loss=0.26571187744140623, metrics={'train_runtime': 912.4306, 'train_samples_per_second': 38.359, 'train_steps_per_second': 4.795, 'total_flos': 9208886937600000.0, 'train_loss': 0.26571187744140623, 'epoch': 1.0})

Evaluate

In [ ]:
preds = trainer.predict(test_dataset) # Get predictions on the test dataset

y_pred = np.argmax(preds.predictions, axis=1) # Get the predicted labels by taking the argmax of the predictions

print("Accuracy:", accuracy_score(test_labels, y_pred)) # Print the accuracy of the model on the test set
print("Confusion Matrix:\n", confusion_matrix(test_labels, y_pred)) # Print the confusion matrix for the test set predictions

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Accuracy: 0.9405333333333333
Confusion Matrix:
 [[3470  252]
 [ 194 3584]]


Experiment 1: Freeze BERT Layers

In [22]:
for param in model.bert.parameters(): # Freeze the BERT layers to prevent them from being updated during training
    param.requires_grad = False # Freeze the BERT layers to prevent them from being updated during training

Experiment 2: Fine-tune Last 2 Layers

In [23]:
for name, param in model.bert.named_parameters(): # Unfreeze the last two layers of BERT to allow them to be updated during training
    if "encoder.layer.10" in name or "encoder.layer.11" in name: # Unfreeze the last two layers of BERT to allow them to be updated during training
        param.requires_grad = True # Unfreeze the last two layers of BERT to allow them to be updated during training
    else:
        param.requires_grad = False # Keep the rest of the BERT layers frozen to prevent them from being updated during training